# 1. Architecture Overview

> **Trust the recommendation because you can trace it, not because you trust the model.**

A multi-agent LLM system for cloud-optimization recommendations, designed around one constraint: every claim in the final report must mechanically resolve to a tool observation. This notebook walks the system end-to-end on one scenario, using artifacts from a real prior Opus run.

**Tour order:** 01 (you're here, ~5 min) → [02 Evaluation & Results](02_Evaluation_and_Results.ipynb) (~8 min) → [03 Harness Design](03_Harness_Design_Deep_Dive.ipynb) (~12 min). Total ~25 min.

Everything in this notebook reads from committed files. **No API key, no network, no LLM cost.** Run cells top to bottom.

## What this project does

Read cloud telemetry across four tiers (compute / database / cache / network), produce an infrastructure-optimization recommendation, attach a complete reasoning trail. The output is structurally auditable: every claim in the final report ties back, by id, to a specific tool observation that produced it.

The system **recommends; it never changes infrastructure state**. A human reviews every recommendation. That boundary is enforced in code, not in prose.

## The problem

Cloud root-cause is hard because the service screaming loudest is rarely the one causing the problem. Application latency spikes look like a compute problem until you find the slow database queries underneath. A single LLM asked to reason across compute, database, and network at once tends to latch onto whichever signal it sees first and produce a plausible-sounding (but wrong) attribution. The full argument for splitting the reasoning is in [`docs/decisions.md`](../docs/decisions.md) — the short version is that bounded specialists with tier-scoped tool surfaces produce diagnoses you can independently verify.

## Six agents in a hierarchy

![Architecture overview — agents in a hierarchy](../docs/images/system.png)

*Rendered diagram; the ASCII version below is the always-renders text fallback (Jupyter image rendering can be theme-dependent).*

```
Trigger
  │
  ▼
Supervisor (router)
  │
  ├──► System Mapper          (parses Terraform → tier topology)
  │
  ├──► Compute Analyst        ┐
  ├──► Data-Layer Analyst     ├─ ReAct loops, parallel, tier-scoped
  ├──► Network Analyst        ┘   (dispatched only if tier present)
  │
  └──► Cross-Tier Evaluator   (reconciles, drift-checks, synthesizes)
       │
       ▼
       Final recommendation → human reviewer
```

The Supervisor is the only router — every worker returns to the Supervisor between stages; no worker decides on its own when the cycle is done. That makes the graph replayable and the audit trail comprehensible.

Wrapping this graph at four points are the **harnesses**: Input (validates the bundle at cycle start), Reasoning (drift-checks each specialist's narrative against its cited evidence), Action (gates the final recommendation), Orchestration (cycle-end consistency checks). They're not in the agent diagram because they're cross-cutting enforcement, not agent steps. Notebook 03 walks them in detail.

## A worked example — scenario 08

Scenario 08 is a *cross-tier* case: a slow set of database queries causes application latency to breach SLA on the compute tier. The naïve diagnosis ("add more compute") would be wrong. The system has to identify the database as the upstream cause and recommend the right fix.

We'll use the vendored telemetry for scenario 08 to make the discussion concrete.

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
SCENARIO_DIR = PROJECT_ROOT / 'dataset-examples' / 'scenario_08'

metadata = json.loads((SCENARIO_DIR / 'metadata.json').read_text())
narrative = metadata['narrative']

print(f"Scenario {metadata['scenario_id']}: {metadata['scenario_name']}")
print(f"Type     : {metadata['scenario_type']}")
print()
print('What this demonstrates:')
print(' ', narrative['what_this_demonstrates'].strip())

Scenario 08: Database Bottleneck Impact
Type     : cross_tier_negative

What this demonstrates:
  A database with slow queries during business hours that cascade into elevated application latency on the compute tier. Compute itself is correctly sized; the problem is downstream.


## How the cycle unfolded — stage by stage

The diagram above shows the agent topology in the abstract. Here it is instantiated for scenario 08 — what each agent in the hierarchy actually produced, pulled from the same `scenario_08_trace.json`. Four stages: **System Mapper** detected which tiers exist; **Supervisor** dispatched specialists in parallel to match; the **Compute** and **Data-Layer Analysts** ran tier-scoped ReAct loops and returned findings; the **Cross-Tier Evaluator** drift-checked, correlated, and synthesized.

This is the bridge between *topology* (the diagram) and *outcome* (the rendered report further down). Every value printed below is read from the bundled trace — no API call.

In [2]:
trace_path = PROJECT_ROOT / 'sample_runs' / 'traces' / 'scenario_08_trace.json'
stage_trace = json.loads(trace_path.read_text())
recon = stage_trace['recommendation']['reconciliation']

# --- Stage 1 — System Mapper ---
print('=== Stage 1 — System Mapper: tier topology + dispatch plan ===')
topo = recon.get('topology_assessment', '') or ''
# The first two sentences carry the high-level dispatch decision.
first_two = '. '.join(topo.split('. ')[:2]).rstrip('.') + '.'
print(f'  {first_two[:420]}')
print()

# --- Stage 2 — Supervisor ---
n_specialists = len(stage_trace['specialist_findings'])
print('=== Stage 2 — Supervisor: dispatch decision ===')
print(f'  Approved parallel dispatch of {n_specialists} specialist(s) matching the active tiers.')
print('  (The Supervisor\'s per-step routing reasoning is persisted in the audit DB')
print('  as supervisor_decision rows; this curated trace carries the outcome —')
print(f'  namely the {n_specialists} specialist_findings entries shown in Stage 3.)')
print()

# --- Stage 3 — Specialists (parallel) ---
print('=== Stage 3 — Specialists: tier-scoped findings, run in parallel ===')
for sf in stage_trace['specialist_findings']:
    f = sf['content']
    headline = (f.get('headline') or '')
    if len(headline) > 200:
        cut = headline.rfind(' ', 0, 200)
        headline = (headline[:cut] + '…') if cut > 0 else headline[:200] + '…'
    print(f"  [{sf['agent']}] finding_type={f.get('finding_type')} / "
          f"primary_tier={f.get('primary_tier')} / confidence={f.get('confidence')}")
    print(f'    {headline}')
print()

# --- Stage 4 — Cross-Tier Evaluator ---
print('=== Stage 4 — Cross-Tier Evaluator: drift-check, correlate, synthesize ===')
for dc in recon.get('drift_check', []):
    print(f"  drift_check[{dc.get('agent')}]: {dc.get('verdict')}")
ctc = recon.get('cross_tier_correlations') or []
if ctc:
    top = ctc[0]
    print(f"  top cross-tier correlation: {top.get('tier_a')} <-> {top.get('tier_b')} "
          f"r={top.get('coefficient')} lag={top.get('lag_minutes')}min")
resolution = recon.get('conflict_resolution', '') or ''
if resolution:
    # Prefer the sentence that names the resolution (starts with 'Resolution:'
    # or contains 'correct'); fall back to the first sentence otherwise.
    sentences = [s.strip() for s in resolution.split('. ') if s.strip()]
    chosen = next(
        (s for s in sentences if s.startswith('Resolution') or 'correct' in s.lower()),
        sentences[0] if sentences else '',
    )
    if chosen and not chosen.endswith('.'):
        chosen += '.'
    print(f'  conflict_resolution: {chosen[:280]}')
comp = stage_trace['recommendation']['composite']
print(f"  synthesized -> primary_tier={comp.get('primary_tier')}, "
      f"secondary_tier={comp.get('secondary_tier')}, "
      f"action_category={comp.get('action_category')}")

=== Stage 1 — System Mapper: tier topology + dispatch plan ===
  Two specialists were invoked: the Compute Analyst and the Data Layer Analyst. No Network Analyst was invoked because the tier topology shows network as null (evidence_ref 424, tier_topology.network: null).

=== Stage 2 — Supervisor: dispatch decision ===
  Approved parallel dispatch of 2 specialist(s) matching the active tiers.
  (The Supervisor's per-step routing reasoning is persisted in the audit DB
  as supervisor_decision rows; this curated trace carries the outcome —
  namely the 2 specialist_findings entries shown in Stage 3.)

=== Stage 3 — Specialists: tier-scoped findings, run in parallel ===
  [compute_analyst] finding_type=issue_found / primary_tier=compute / confidence=0.97
    Severe P95 latency SLA breach during weekday business hours: latency reaches 620-670ms against a 300ms SLA target on a tier-1 checkout service.
  [data_layer_analyst] finding_type=issue_found / primary_tier=database / confidence=0.97
 

## What the system produced

Those four stages compose into a single recommendation. Below are the taxonomy fields the Cross-Tier Evaluator assembled — the actual output for this cycle, loaded from `sample_runs/traces/scenario_08_trace.json`. The cross-tier diagnosis line that follows is derived from those fields, not narrated separately.

In [3]:
trace = json.loads(
    (PROJECT_ROOT / 'sample_runs' / 'traces' / 'scenario_08_trace.json').read_text()
)

print(f"cycle_id       : {trace['cycle_id']}")
print(f"application_id : {trace['application_id']}")
print()

composite = trace['recommendation']['composite']
primary   = composite.get('primary_tier')
secondary = composite.get('secondary_tier')

print('--- final recommendation (taxonomy fields, derived from the trace) ---')
for field in ('finding_type', 'primary_tier', 'secondary_tier', 'action_category'):
    print(f'  {field:18s}: {composite.get(field)!r}')
print()
print(f'Cross-tier diagnosis: {primary} is the upstream cause; {secondary} is downstream.')
print()
print(f"Specialists that fired in this cycle: {len(trace['specialist_findings'])}")
for sf in trace['specialist_findings']:
    print(f"  - {sf['agent']}")
print()
print(f"Traceability check: {trace['summary']['unique_refs_cited']} evidence references cited; "
      f"{trace['summary']['dangling']} dangling.")

cycle_id       : cycle_20260604_150610_37995130
application_id : app-08

--- final recommendation (taxonomy fields, derived from the trace) ---
  finding_type      : 'issue_found'
  primary_tier      : 'database'
  secondary_tier    : 'compute'
  action_category   : 'query_cache_optimization'

Cross-tier diagnosis: database is the upstream cause; compute is downstream.

Specialists that fired in this cycle: 2
  - compute_analyst
  - data_layer_analyst

Traceability check: 26 evidence references cited; 0 dangling.


## The full rendered report

The same audit trail rendered into a human-facing markdown report — the artifact a reviewer would actually read. Recommendation, reasoning, specialist findings, cross-tier analysis, trade-offs, evaluator confidence.

In [4]:
from IPython.display import HTML, display
from markdown_it import MarkdownIt

report_text = (PROJECT_ROOT / 'sample_runs' / 'reports' / 'scenario_08_report.md').read_text()

# Two problems to solve at once:
#   1. JupyterLab's MathJax 3 reads `$...$` pairs as inline math, which mangles
#      dollar amounts in the trade-off table (Value+Note columns visually merge).
#   2. The IPython Markdown processor doesn't fully handle raw HTML inside GFM
#      table cells, so wrapping each `$` in a `<span>` leaks closing tags as text.
# Fix: render the markdown to HTML with markdown-it-py (already shipped with
# IPython), then wrap the whole rendered block in a single MathJax-3 opt-out
# div. No HTML touches the table cells; the wrapper attribute tells MathJax to
# skip the entire subtree.
md = MarkdownIt('commonmark').enable('table')
html_body = md.render(report_text)
display(HTML(
    '<div class="tex2jax_ignore mathjax_ignore" data-mjx-process="false">'
    f'{html_body}'
    '</div>'
))

## Four things to notice

If you're skimming for engineering signals, these are visible above:

1. **Specialists are tier-scoped, not omniscient.** Each ReAct loop sees only its own tier's tools — Compute Analyst can't query the database. Scope is enforced at the tool surface, not by asking the agent nicely. This is what makes the cross-tier drift-check meaningful: the Evaluator can't be fooled into thinking specialists agreed when they only saw each other's data.
2. **The Supervisor is the only router.** No worker decides when it's done; control returns to the Supervisor between every stage. The graph is replayable; the audit trail is comprehensible.
3. **Evidence-binding is structural, not stylistic.** Every claim in the rendered report cites a numeric `evidence_ref` resolving to an audit-DB row. The Action Harness refuses to surface a recommendation with dangling refs — a missing citation is a gate failure, not a polish issue.
4. **The system recommends; it never acts.** No infrastructure mutation is built into the harness. A human reviews every recommendation. The boundary is enforced in code, not in prose.

## Continue the tour

→ **[02 Evaluation & Results](02_Evaluation_and_Results.ipynb)** — how the system is scored. Four-layer eval, 18 scenarios, published per-model-tier baselines.

→ **[03 Harness Design](03_Harness_Design_Deep_Dive.ipynb)** — how correctness is enforced. The four harnesses, the backward evidence-chain walk, the mechanical proof of zero dangling references.

For the live experience, see [`docs/running.md`](../docs/running.md): `make scenario APP=app-08` runs the agents end-to-end on your machine; `docker compose up --build live-llm` does the same hermetic in Docker.